In [ ]:
# Installation des packages
# %pip install numpy pandas matplotlib seaborn scikit-learn joblib

In [1]:
## Packages
import numpy as np
import pandas as pd

# Chemin de sortie des CSV
SORTIE = "../Travail Julie/"

In [ ]:
## Importation des données
# Sur Google colab (sur le drive) :
# from google.colab import drive
# drive.mount('/content/drive')
# directory = '/content/drive/MyDrive/Datacamp/'

# Chemin relatif : on remonte d'un dossier (Travail Julie -> Datacamp) pour
# accéder aux fichiers de données.
directory_data = '../'

x = pd.read_csv(directory_data + 'engie_X.csv', header=0, index_col=None, sep=';', decimal='.')
y = pd.read_csv(directory_data + 'engie_Y.csv', header=0, index_col=None, sep=';', decimal='.')

x.info()
x.head()

In [ ]:
y.info()
y.head()

In [ ]:
## Analyse des données manquantes

# On concatène les données pour éviter les erreurs de traitement
data = pd.merge(x, y, on='ID', how='inner')
print('Dimensions du jeu de données :', data.shape)

missing_percentage = data.isna().mean() * 100
print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))


In [ ]:

# On constate que Grid_voltage, Grid_voltage_min, Grid_voltage_max et Grid_voltage_std
# présentent beaucoup de données manquantes (environ 16%) ; on analyse ces variables
# pour décider de leur retrait ou de leur imputation.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(data['Grid_voltage'], data['TARGET'])
ax.grid()
ax.set_xlabel('Grid_voltage')
ax.set_ylabel('TARGET')
plt.show()


In [ ]:
# Pas de corrélation visible avec TARGET -> on retire ces variables :
to_remove = data.columns[data.columns.str.startswith('Grid_voltage')]
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

In [ ]:
## Restriction du champ d'étude
# Absolute_wind_direction / Absolute_wind_direction_c et Nacelle_angle / Nacelle_angle_c
# sont quasiment identiques (seules des valeurs manquantes les séparent), on en
# supprime une sur deux (celle ne présentant pas de données manquantes par hypothèse, 
# il faudrait vérifier la pertinence de cette dernière auprès du "métier") :
different_rows = data[['Absolute_wind_direction', 'Absolute_wind_direction_c']][
    data['Absolute_wind_direction'] != data['Absolute_wind_direction_c']]
print('Nombre de lignes différentes :', different_rows.shape[0])
different_rows.head()

different_rows = data[['Nacelle_angle', 'Nacelle_angle_c']][
    data['Nacelle_angle'] != data['Nacelle_angle_c']]
print('Nombre de lignes différentes :', different_rows.shape[0])
different_rows.head()

to_remove = ['Absolute_wind_direction_c', 'Nacelle_angle_c']
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

In [ ]:
# On retire les observations pour lesquelles il subsiste des données manquantes :
data.dropna(inplace=True)
print('Dimensions du jeu de données :', data.shape)

data['MAC_CODE'].value_counts()

In [ ]:
# Variables d'angles - variable circulaire, on transforme en cos/sin pour éviter
# les effets de bord (0° = 360°) :
data['Nacelle_angle_cos'] = np.cos(np.cos(data['Nacelle_angle']))
data['Nacelle_angle_sin'] = np.cos(np.sin(data['Nacelle_angle']))
data['Nacelle_angle_min_cos'] = np.cos(np.cos(data['Nacelle_angle_min']))
data['Nacelle_angle_min_sin'] = np.cos(np.sin(data['Nacelle_angle_min']))
data['Nacelle_angle_max_cos'] = np.cos(np.cos(data['Nacelle_angle_max']))
data['Nacelle_angle_max_sin'] = np.cos(np.sin(data['Nacelle_angle_max']))

to_remove = ['Nacelle_angle', 'Nacelle_angle_min', 'Nacelle_angle_max']
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

data['Pitch_angle_cos'] = np.cos(np.cos(data['Pitch_angle']))
data['Pitch_angle_sin'] = np.cos(np.sin(data['Pitch_angle']))
data['Pitch_angle_min_cos'] = np.cos(np.cos(data['Pitch_angle_min']))
data['Pitch_angle_min_sin'] = np.cos(np.sin(data['Pitch_angle_min']))
data['Pitch_angle_max_cos'] = np.cos(np.cos(data['Pitch_angle_max']))
data['Pitch_angle_max_sin'] = np.cos(np.sin(data['Pitch_angle_max']))

to_remove = ['Pitch_angle', 'Pitch_angle_min', 'Pitch_angle_max']
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

data['Absolute_wind_direction_cos'] = np.cos(np.cos(data['Absolute_wind_direction']))
data['Absolute_wind_direction_sin'] = np.cos(np.sin(data['Absolute_wind_direction']))
to_remove = ['Absolute_wind_direction']
data = data.drop(columns=to_remove)
print('Dimensions du jeu de données :', data.shape)

In [ ]:
# On travaille sur la turbine n°4 :
data = data[data['MAC_CODE'] == 'WT4']
print('Dimensions du jeu de données :', data.shape)

In [ ]:
# Lien entre TARGET et Rotor_speed :
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(data['Rotor_speed'], data['TARGET'])
ax.grid()
ax.set_xlabel('Rotor_speed')
ax.set_ylabel('TARGET')
plt.show()

In [ ]:
# On retire les puissances nulles ou négatives :
data = data[data['TARGET'] > 0]
print('Dimensions du jeu de données :', data.shape)

In [ ]:
# Sous-échantillonnage à 10% pour accélérer les calculs
sample_frac = 1/10
data = data.sample(frac=sample_frac, random_state=1789)
print('Dimensions du jeu de données :', data.shape)

In [ ]:
## Sauvegarde pour les fichiers 02 (exploration) et 03 (modélisation)
data.to_csv(SORTIE + 'engie_data_clean.csv', index=False, sep=';', decimal='.')
print('Fichier sauvegardé :', SORTIE + 'engie_data_clean.csv')